# Vertex AI Model Registry: Register, Version, and Alias

The model was already trained in the previous lecture (L4) using a custom Docker container.
The trained artifact (model.joblib) is sitting in our GCS bucket.

In this notebook we:
1. Register the trained model to the Model Registry (Version 1)
2. Assign an alias to identify it
3. Upload a second version of the same model
4. List and compare versions
5. Retrieve a specific version by alias

In [ ]:
from google.cloud import aiplatform

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"      # Replace with your GCP project ID
REGION = "us-central1"
BUCKET_NAME = "YOUR_BUCKET_NAME"    # Replace with your GCS bucket name

aiplatform.init(project=PROJECT_ID, location=REGION)

## 1. Register the Trained Model (Version 1)

In [ ]:
model_v1 = aiplatform.Model.upload(
    display_name="bikeshare-model",
    artifact_uri=f"gs://{BUCKET_NAME}/bikeshare-artifact/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
)

model_v1.wait()
print(f"Model registered: {model_v1.display_name}")
print(f"Resource name: {model_v1.resource_name}")
print(f"Version ID: {model_v1.version_id}")

## 2. Assign an Alias to Version 1
Aliases are human-readable labels you attach to a version. Use them to track which version is in production, which is being tested, etc.

In [ ]:
registry = aiplatform.models.ModelRegistry(model=model_v1.resource_name)

registry.add_version_aliases(
    new_aliases=["champion"],
    version=model_v1.version_id,
)
print(f"Alias 'champion' assigned to version {model_v1.version_id}")

## 3. Upload a New Version of the Same Model (Version 2)
Use the parent_model parameter to register a new version under the same model entry instead of creating a separate model.

In [ ]:
model_v2 = aiplatform.Model.upload(
    display_name="bikeshare-model",
    artifact_uri=f"gs://{BUCKET_NAME}/bikeshare-artifact/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
    parent_model=model_v1.resource_name,
    is_default_version=False,
)

model_v2.wait()
print(f"Version 2 registered: {model_v2.version_id}")

In [ ]:
registry.add_version_aliases(
    new_aliases=["challenger"],
    version=model_v2.version_id,
)
print(f"Alias 'challenger' assigned to version {model_v2.version_id}")

## 4. List All Versions

In [ ]:
versions = registry.list_versions()

for v in versions:
    print(f"Version: {v.version_id} | Aliases: {v.version_aliases} | Created: {v.version_create_time}")

## 5. Retrieve a Model by Alias
Instead of remembering version numbers, you can load a model directly by its alias.

In [ ]:
champion_model = registry.get_model(version="champion")
print(f"Champion model version: {champion_model.version_id}")
print(f"Resource name: {champion_model.resource_name}")

## 6. Promote Challenger to Champion
When the new version performs better, swap the aliases.

In [ ]:
registry.remove_version_aliases(
    target_aliases=["champion"],
    version=model_v1.version_id,
)

registry.add_version_aliases(
    new_aliases=["champion"],
    version=model_v2.version_id,
)

registry.remove_version_aliases(
    target_aliases=["challenger"],
    version=model_v2.version_id,
)

print("Promotion complete. Listing updated versions:")
for v in registry.list_versions():
    print(f"Version: {v.version_id} | Aliases: {v.version_aliases}")